##DATA PREPARATION

Note: For clarity, explanations are included after each code block and graph, particularly in the EDA and hypothesis testing sections.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats


mortality = pd.read_excel("data/UNIGME-2025-UNICEFRegion-Rates-Deaths-Sex-specific-5-to-24.xlsx",sheet_name="20q5, 10q10",skiprows=14)

mortality.columns = mortality.columns.str.strip()

mortality = mortality[["Region", "Sex", "Year", "Median"]] #data analizi için only relevant variablesları seçtim

mortality = mortality.rename(columns={
    "Median": "mort_median_5_24"
})

mortality = mortality.groupby(["Region", "Year"])["mort_median_5_24"].mean().reset_index()
# veriyi region + year bazında ortalama alarak özetliyoruz, ayrıca bu adımla sex ve region level kısmını datasetten çıkardık
hdi = pd.read_excel(
    "data/HDR25_Statistical_Annex_HDI_Table.xlsx",
    skiprows=4,
    usecols="B:C"
)




mapping = pd.read_excel("data/JME_Regional-Classifications.xlsx")

mapping = mapping[["Country", "UNICEF Region"]]
#mapping = mapping.rename(columns={"UNICEF Region": " UNICEF Region"})
mapping.head()



,Country,UNICEF Region
0,Afghanistan,ROSA
1,Albania,ECA
2,Algeria,MENA
3,Andorra,ECA
4,Angola,SSA


In [2]:
econ = pd.read_excel(
    "data/19-Economic-Indicators-SOWC2025-1.xlsx",
    skiprows=6,
    header=None,
    usecols=[1, 6, 8]
)

econ.columns = ["Country", "health_spending", "education_spending"]

econ = econ.iloc[1:].reset_index(drop=True)

econ["health_spending"] = pd.to_numeric(econ["health_spending"], errors="coerce")
econ["education_spending"] = pd.to_numeric(econ["education_spending"], errors="coerce")

econ = econ.merge(mapping, on="Country", how="left")#burada econ datasındaki ülkelere, mapping datasından hangi bölgeye ait olduklarını ekledik
econ.head()

,Country,health_spending,education_spending,UNICEF Region
0,Afghanistan,0.720916,4.34319,ROSA
1,Albania,2.884195,2.74931,ECA
2,Algeria,3.267783,6.29900,MENA
3,Andorra,6.167981,2.66623,ECA
4,Angola,1.711142,2.33200,SSA


In [3]:
hdi.columns = ["Country", "HDI"]
hdi = hdi.iloc[2:].reset_index(drop=True)#gereksiz satırları silip index’i resetledim
hdi.head()
hdi = hdi.merge(mapping, on="Country", how="left")#burada da HDI verisindeki ülkelere hangi bölgeye ait olduklarını ekledik
hdi.head()

,Country,HDI,UNICEF Region
0,Very high human development,NaN,NaN
1,Iceland,0.972,ECA
2,Norway,0.97,ECA
3,Switzerland,0.97,ECA
4,Denmark,0.962,ECA


In [4]:
hdi = hdi.rename(columns={"UNICEF Region": "Region"})
hdi["HDI"] = pd.to_numeric(hdi["HDI"], errors="coerce")

hdi_region = hdi.groupby("Region")["HDI"].mean().reset_index()#ülkeleri bölgelere göre gruplayıp ortalama HDI hesaplıyoruz
hdi_region.head(10)

,Region,HDI
0,EAPRO,0.735857
1,ECA,0.882796
2,LACRO,0.768424
3,MENA,0.776778
4,ROSA,0.659000
5,SSA,0.561587


In [5]:
region_map = {
    "EAPRO": "East Asia and Pacific",
    "ECA": "Europe and Central Asia",
    "LACRO": "Latin America and Caribbean",
    "MENA": "Middle East and North Africa",
    "ROSA": "South Asia",
    "SSA": "Sub-Saharan Africa"
}
#burada kısaltılmış region kodlarını tam isimlere çevirmek için mapping oluşturdum(gpt söyledi)
hdi_region["Region"] = hdi_region["Region"].replace(region_map)
hdi_region

,Region,HDI
0,East Asia and Pacific,0.735857
1,Europe and Central Asia,0.882796
2,Latin America and Caribbean,0.768424
3,Middle East and North Africa,0.776778
4,South Asia,0.659000
5,Sub-Saharan Africa,0.561587


In [33]:
econ_region = econ.groupby("UNICEF Region")[["health_spending", "education_spending"]].mean().reset_index()
econ_region["UNICEF Region"] = econ_region["UNICEF Region"].replace(region_map)
econ_region = econ_region.rename(columns={"UNICEF Region": "Region"})
econ_region.head()

,Region,health_spending,education_spending
0,East Asia and Pacific,4.563424,4.522219
1,Europe and Central Asia,5.864425,4.702014
2,Latin America and Caribbean,4.406373,4.369904
3,Middle East and North Africa,3.170344,4.231401
4,South Asia,2.020454,3.786024


Ülkeleri bölgelere göre gruplayıp: health spending ortalaması ve education spending ortalaması aldık
(groupby ile)

In [8]:
df = mortality.merge(hdi_region, on="Region", how="left") #burada mortality datasına hdi ekledim
df = df.merge(econ_region, on="Region", how="left")#burada da economic verileri ekledim

df.head()

,Region,Year,mort_median_5_24,HDI,health_spending,education_spending
0,East Asia and Pacific,1990,18.477822,0.735857,4.563424,4.522219
1,East Asia and Pacific,1991,18.001312,0.735857,4.563424,4.522219
2,East Asia and Pacific,1992,17.444175,0.735857,4.563424,4.522219
3,East Asia and Pacific,1993,16.978292,0.735857,4.563424,4.522219
4,East Asia and Pacific,1994,16.549464,0.735857,4.563424,4.522219


In [9]:
df.isnull().sum()#merge sonrası verileri kontrol eder her sütun biribriyle eşleniyor mu boş veri var mı diye

Region                  0
Year                    0
mort_median_5_24        0
HDI                   210
health_spending       210
education_spending    210
dtype: int64

In [10]:
fix_map = {
    "Eastern Europe and Central Asia": "Europe and Central Asia",
    "Western Europe": "Europe and Central Asia",
    "Eastern and Southern Africa": "Sub-Saharan Africa",
    "West and Central Africa": "Sub-Saharan Africa",
    "North America": "Europe and Central Asia",  # approx (dataset'e göre kabul)
    "World": None  # bunu istemiyoruz
}

df["Region"] = df["Region"].replace(fix_map)#mortality datasetindeki region isimlerini diğer datasetlerle uyumlu hale getirdik

In [34]:
df = mortality.copy()
df["Region"] = df["Region"].replace(fix_map)

df = df.merge(hdi_region, on="Region", how="left")
df = df.merge(econ_region, on="Region", how="left")
df.isnull().sum()

Region                35
Year                   0
mort_median_5_24       0
HDI                   35
health_spending       35
education_spending    35
dtype: int64

In [35]:
df = df[df["Region"].notna()].reset_index(drop=True)# burada region değeri boş (NaN) olan satırları sildik, sonra index’i sıfırlayıp düzenledik
df.isnull().sum()

Region                0
Year                  0
mort_median_5_24      0
HDI                   0
health_spending       0
education_spending    0
dtype: int64

###EXPLORATORY DATA ANALYSIS AND HYPOTHESIS TESTING – YOUTH MORTALITY AND SOCIO-ECONOMIC FACTORS


This part of the notebook presents the main exploratory data analysis (EDA) and hypothesis testing for the project.

**Understanding Youth Mortality: The Role of Socio-Economic Development Across Regions**

The goal of this analysis is to explore how youth mortality rates (ages 5–24) are related to socio-economic factors such as the Human Development Index (HDI), health spending, and education spending. In addition, the analysis examines regional differences in mortality outcomes.

The notebook is organised as follows:

- Data loading, cleaning, and merging from multiple sources  
- Descriptive statistics and data validation  
- Relationship between HDI and mortality (H1)  
- Statistical testing (correlation, t-test, ANOVA)  
- Economic indicators and mortality (H2)  
- Regional differences in mortality rates (H3)
	

In [15]:
df.describe()#burada da tüm data cleaning işlemleri sonucundaki özet istatistikleri veriyor

,Year,mort_median_5_24,HDI,health_spending,education_spending
count,385.000000,385.000000,385.000000,385.000000,385.000000
mean,2007.000000,24.998897,0.741455,3.919837,4.316662
std,10.112647,21.123060,0.130124,1.734485,0.356637
min,1990.000000,3.720566,0.561587,1.833304,3.786024
25%,1998.000000,9.734127,0.561587,1.833304,3.921891
50%,2007.000000,15.814418,0.768424,4.406373,4.369904
75%,2016.000000,36.756472,0.882796,5.864425,4.702014
max,2024.000000,117.836929,0.882796,5.864425,4.702014


In [32]:
import statsmodels.formula.api as smf

model = smf.ols(
    "mort_median_5_24 ~ HDI + health_spending + education_spending + C(Region)",
    data=df
).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:       mort_median_5_24   R-squared:                       0.785
Model:                            OLS   Adj. R-squared:                  0.782
Method:                 Least Squares   F-statistic:                     276.3
Date:                Fri, 01 May 2026   Prob (F-statistic):          5.68e-124
Time:                        14:07:33   Log-Likelihood:                -1424.6
No. Observations:                 385   AIC:                             2861.
Df Residuals:                     379   BIC:                             2885.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                                                coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------------


## Regression Summary (HDI + Spending + Region)

 **Model Performance**
* R² ≈ 0.78 → strong explanatory power
* Model is statistically significant (p < 0.001)

👉 The model explains most of the variation in youth mortality.

⸻
 **Health Spending**

* Negative and significant effect (p < 0.001)

👉 Higher health spending is associated with lower youth mortality
👉 Most reliable and clear predictor

⸻
** Education Spending**

* Positive and significant coefficient (p < 0.001)

⚠️ Interpretation should be cautious
👉 Likely affected by multicollinearity

👉 Does NOT mean education increases mortality
👉 Indicates overlapping effects with other variables

⸻

** HDI**

* Not statistically significant (p > 0.05)

👉 When more specific variables are included,
HDI’s independent effect becomes unclear

👉 Suggests HDI overlaps with spending variables

⸻

**Regional Effects**

* Several regions remain statistically significant

👉 Even after controlling for HDI and spending:
mortality differs across regions

👉 Indicates presence of structural / geographic factors

⸻

* Multicollinearity*

* Strong correlation between HDI, health, and education

👉 Makes it difficult to isolate individual effects
👉 Coefficients should be interpreted carefully

⸻

*Final Insight*

👉 Youth mortality is not explained by development alone

👉 It is shaped by:

* targeted investments (especially health)
* regional structural differences

